In [1]:
import allantools
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import scienceplots
from scipy.optimize import curve_fit
import os 
from math import pi
plt.style.use('science')
from tqdm import tqdm

Defining functions

In [2]:
#general sine curve model
def model(t, f, p, c, a):
    return a*np.sin(2*pi*f*t-p)+c

#Take a dataframe and do the fitting.
def split_n_fit(df, f):
    time = df['time'].values
    signal = df['signal'].values

    offset_guess = np.mean(signal)
    amplitude_guess = (np.max(signal) - np.min(signal)) / 2

    try:
        popt, _ = curve_fit(
            model, time, signal,
            p0=[f, 1, offset_guess, amplitude_guess],
            maxfev=1000, xtol=1e-12, ftol=1e-12
        )

        # Normalize amplitude and phase
        fitted_freq = popt[0]
        fitted_phase = popt[1]
        fitted_offset = popt[2]
        fitted_amplitude = popt[3]

        if fitted_amplitude < 0:
            fitted_amplitude = -fitted_amplitude
            fitted_phase -= pi

        # Create result in the same shape as before
        result = pd.DataFrame([[fitted_freq, fitted_phase, fitted_offset, fitted_amplitude]])
        result.columns = ['frequency', 'phase', 'offset', 'amplitude']
        return result.describe().T
    except RuntimeError:
        print('Fit error')
        # Return NaNs but still with the same structure
        result = pd.DataFrame([[np.nan, np.nan, np.nan, np.nan]])
        result.columns = ['frequency', 'phase', 'offset', 'amplitude']
        return result.describe().T



#read each file in a given folder name
def read(folder_path):
    dataframes = []
    if not os.path.isdir(folder_path):
        print(f"Folder '{folder_path}' does not exist.")
        return dataframes
    for filename in tqdm(os.listdir(folder_path)):
        file_path = os.path.join(folder_path, filename)
        if os.path.isfile(file_path):
            try:
                df = pd.read_csv(file_path, delimiter='\t')
                dataframes.append(df)
            except Exception as e:
                print(f"Could not read {filename}: {e}")
    return dataframes

#modify the dataframe
def cleaning(df):
    data_frames = []
    if df.iloc[0,0] == '(ns)':
        timescale = 1e-9
    elif df.iloc[0,0] == '(us)':
        timescale = 1e-6
    elif df.iloc[0,0] == '(ms)':
        timescale = 1e-3
    elif df.iloc[0,0] == '(s)':
        timescale = 1
    df = df.drop(index=0)
    df = df.astype(float)
    df_1 = df.copy()
    df_2 = df.copy()
    df_1.columns = ['time', 'signal', 'x']
    df_1['time'] = df_1['time']*timescale
    data_frames.append(df_1)
    df_2.columns = ['time', 'x', 'signal']
    df_2['time'] = df_2['time']*timescale
    data_frames.append(df_2)
    return data_frames

#summarize result from split_n_fit function
def summarize(result):
    freq = result.iloc[1, 0]
    phase = result.iloc[1, 1]
    df = pd.DataFrame([freq, phase])
    return df

def f(folder_name, frequency):
    dfs = read(folder_name)

    fit_result_A_list = []
    fit_result_B_list = []

    for i in tqdm(dfs):
        data_frames = cleaning(i)
        result_A = summarize(split_n_fit(data_frames[0], frequency))
        result_B = summarize(split_n_fit(data_frames[1], frequency))
        fit_result_A_list.append(result_A)
        fit_result_B_list.append(result_B)

    # Do pd.concat once, after the loop
    fit_result_A = pd.concat(fit_result_A_list, axis=1)
    fit_result_B = pd.concat(fit_result_B_list, axis=1)

    result = pd.concat([fit_result_A.T, fit_result_B.T], axis=1)
    result.columns = ['frequency A', 'phase A', 'frequency B', 'phase B']
    phase_diff = result['phase B'] - result['phase A']
    return np.unwrap(phase_diff) / frequency


def allan_deviation(phase_diff, rate, data_type):
    t, ad, ade, adn = allantools.oadev(phase_diff, rate = rate, data_type = data_type)
    fig, ax = plt.subplots(1,1, figsize=(4,4),dpi=300, constrained_layout=True)
    ax.grid()
    ax.loglog(t, ad, marker = 'o', ms = 5, color='k')
    ax.set_xlabel(r'Averaging time [s]')
    ax.set_ylabel(r'$\sigma(\tau)$')
    ax.set_title(r'Allan Deviation')